In [1]:
import os
import glob
import csv
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image

In [6]:
# ==========================================
# CONFIGURATION
# ==========================================
# Path to your BraTS 2023 GLI Training Data
DATASET_ROOT = "./ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

# Output folder name
OUTPUT_ROOT = "./2D_Scan"

# How many patients to process?
NUM_SCANS = 5

# File suffixes (Input from NIfTI) -> Output Suffix for JPG
# We map the internal logic to the specific output filename you requested
SUFFIX_MAP = {
    't1':   { 'in': '-t1n.nii.gz', 'out': '-t1n' },
    't1ce': { 'in': '-t1c.nii.gz', 'out': '-t1c' },
    't2':   { 'in': '-t2w.nii.gz', 'out': '-t2w' },
    'flair':{ 'in': '-t2f.nii.gz', 'out': '-t2f' },
    'mask': { 'in': '-seg.nii.gz', 'out': '-seg' }
}

In [7]:
def normalize_to_jpg(img_data):
    """
    Normalizes MRI intensity (0-max) to JPEG range (0-255).
    """
    if np.max(img_data) == 0:
        return img_data.astype(np.uint8)
    
    # Clip top 1% outliers to improve contrast
    p99 = np.percentile(img_data, 99)
    img_data = np.clip(img_data, 0, p99)
    
    # Min-Max Normalization
    img_norm = (img_data - np.min(img_data)) / (np.max(img_data) - np.min(img_data))
    img_norm = (img_norm * 255).astype(np.uint8)
    
    return img_norm

def find_best_slice(seg_volume):
    """
    Finds the index of the axial slice with the largest tumor area.
    """
    tumor_counts = np.sum(seg_volume > 0, axis=(0, 1))
    max_tumor_pixels = np.max(tumor_counts)
    
    if max_tumor_pixels == 0:
        return seg_volume.shape[2] // 2, False
    else:
        return np.argmax(tumor_counts), True

def process_dataset():
    if not os.path.exists(OUTPUT_ROOT):
        os.makedirs(OUTPUT_ROOT)

    # Get sorted list of patient folders
    all_patients = sorted([f for f in os.listdir(DATASET_ROOT) 
                           if os.path.isdir(os.path.join(DATASET_ROOT, f))])
    
    patients_to_process = all_patients[:NUM_SCANS]
    
    print(f"Processing {len(patients_to_process)} patients...")

    csv_data = []
    csv_headers = ['Patient_ID', 'Best_Slice_Index', 'Tumor_Found', 'Output_Path']

    for patient_id in patients_to_process:
        patient_path = os.path.join(DATASET_ROOT, patient_id)
        print(f"Processing: {patient_id}...", end=" ")

        # 1. Load Mask to find Best Slice
        mask_config = SUFFIX_MAP['mask']
        mask_path = os.path.join(patient_path, f"{patient_id}{mask_config['in']}")
        
        if not os.path.exists(mask_path):
            print("SKIPPING (Mask missing)")
            continue

        mask_img = nib.load(mask_path)
        mask_data = mask_img.get_fdata()
        best_slice_idx, tumor_found = find_best_slice(mask_data)

        # 2. Create Patient Subfolder
        patient_out_dir = os.path.join(OUTPUT_ROOT, patient_id)
        if not os.path.exists(patient_out_dir):
            os.makedirs(patient_out_dir)

        # 3. Process All Files
        for key, config in SUFFIX_MAP.items():
            input_filename = f"{patient_id}{config['in']}"
            file_path = os.path.join(patient_path, input_filename)
            
            if os.path.exists(file_path):
                img = nib.load(file_path)
                data_3d = img.get_fdata()
                
                # Extract Slice & Rotate
                slice_2d = data_3d[:, :, best_slice_idx]
                slice_2d = np.rot90(slice_2d)

                # Normalize
                if key == 'mask':
                    # Scale mask (0,1,2,4) to visible range (0-255)
                    slice_img_array = (slice_2d * (255/4)).astype(np.uint8)
                else:
                    slice_img_array = normalize_to_jpg(slice_2d)

                # Construct Specific Output Filename
                # Example: BraTS-GLI-00000-000-t1n.jpg
                output_filename = f"{patient_id}{config['out']}.jpg"
                save_full_path = os.path.join(patient_out_dir, output_filename)
                
                im = Image.fromarray(slice_img_array)
                im.save(save_full_path)

        print(f"Done (Slice {best_slice_idx})")

        csv_data.append({
            'Patient_ID': patient_id,
            'Best_Slice_Index': best_slice_idx,
            'Tumor_Found': tumor_found,
            'Output_Path': patient_out_dir
        })

    # Save Log
    with open(os.path.join(OUTPUT_ROOT, 'dataset_log.csv'), 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()
        writer.writerows(csv_data)

    print("Processing Complete.")

if __name__ == "__main__":
    process_dataset()
    

Processing 5 patients...
Processing: BraTS-GLI-00000-000... Done (Slice 74)
Processing: BraTS-GLI-00002-000... Done (Slice 78)
Processing: BraTS-GLI-00003-000... Done (Slice 109)
Processing: BraTS-GLI-00005-000... Done (Slice 100)
Processing: BraTS-GLI-00006-000... Done (Slice 69)
Processing Complete.
